In [ ]:
!pip install -q torch
!pip install -q torchvision==0.20.0

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.optim.lr_scheduler import StepLR

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [ ]:
def train(model, device, train_loader, optimizer, epoch):
    train_losses = []
    train_sizes = []
    model.train()
    count = (epoch - 1) * len(train_loader.dataset)
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        count += len(data)
        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            train_losses.append(loss.item())
            train_sizes.append(count)
            
            # torch.save(network.state_dict(), '/model/model.pth')
            # torch.save(optimizer.state_dict(), '/model/optimizer.pth')
    return train_sizes, train_losses

In [ ]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))
    return test_loss

In [ ]:
transform=torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.1307,), (0.3081,))
        ])
dataset1 = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
dataset2 = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)

dataset1, dataset2

In [ ]:
train_loader = torch.utils.data.DataLoader(dataset1, batch_size = 64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset2, batch_size = 1000, shuffle=True)

In [ ]:
device = torch.device("cpu")
model = Net().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=1.0)
scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
epochs = 3

In [ ]:
train_losses = []
train_sizes = []

test_losses = []
test_sizes = [i*len(train_loader.dataset) for i in range(epochs + 1)]

test_loss = test(model, device, test_loader)
test_losses.append(test_loss)

for epoch in range(1, epochs + 1):

        sizes, losses = train(model, device, train_loader, optimizer, epoch)
        train_losses.extend(losses)
        train_sizes.extend(sizes)
    
        test_loss = test(model, device, test_loader)
        test_losses.append(test_loss)
    
        scheduler.step()

# if args.save_model:
#     torch.save(model.state_dict(), "mnist_cnn.pt")
# network_state_dict = torch.load("mnist_cnn.pt")
# continued_network.load_state_dict(network_state_dict)

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure()
plt.plot(train_sizes, train_losses, color='blue')
plt.scatter(test_sizes, test_losses, color='red')
plt.legend(['Train Loss', 'Test Loss'], loc='upper right')
plt.xlabel('number of training examples seen')
plt.ylabel('negative log likelihood loss')

In [ ]:
examples = enumerate(test_loader)
batch_idx, (example_data, example_targets) = next(examples)

batch_idx, example_data.shape, example_targets.shape, len(example_targets)

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(12, 2))
for i in range(6):
  plt.subplot(1, 6, i+1)
  plt.tight_layout()
  plt.imshow(example_data[i][0], cmap='gray', interpolation='none')
  plt.title("Ground Truth: {}".format(example_targets[i]))
  plt.xticks([])
  plt.yticks([])

In [ ]:
with torch.no_grad():
  output = model(example_data)
print(output.shape)

fig = plt.figure(figsize=(12, 2))
for i in range(6):
  plt.subplot(1,6,i+1)
  plt.tight_layout()
  plt.imshow(example_data[i][0], cmap='gray', interpolation='none')
  plt.title("Prediction: {}".format(output.data.max(1, keepdim=True)[1][i].item()))
  plt.xticks([])
  plt.yticks([])